<a href="https://colab.research.google.com/github/Oruntu-Tanima-Proje/otProje/blob/main/notebooks/03_mobilenetv2_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 03 - MobileNetV2 Eğitimi

## Hafif ve Mobil-Uyumlu Model

Bu notebook, **MobileNetV2** mimarisini transfer learning ile domates yaprağı
hastalık sınıflandırma problemine uyarlar.

### Model Özellikleri
- **Mimari**: MobileNetV2 (Google, 2018)
- **Parametre**: ~2.4 milyon
- **Boyut**: ~22 MB
- **Avantaj**: Hafif, mobil cihazlarda çalışabilir

### Eğitim Stratejisi
1. **Phase 1**: Feature Extraction — Base donmuş, sadece üst katmanlar eğitilir (15 epoch)
2. **Phase 2**: Fine-Tuning — Son 30 katman açılır, düşük LR ile ince ayar (10 epoch)

### Beklenen Sonuç
- Test Accuracy: ~%92
- Eğitim Süresi: ~90 dakika (T4 GPU)

In [ ]:
# ============================================================
# 1. HAZIRLIK - Drive bağla, veri ve sabitleri ayarla
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import tensorflow as tf

# Sabit ayarlar
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 10
drive_proje = "/content/drive/MyDrive/Domates_Projesi"

# Veriyi Drive'dan kopyala (yoksa)
if not os.path.exists("tomato_data"):
    print("📦 Veri seti Drive'dan kopyalanıyor...")
    shutil.copytree(f"{drive_proje}/data", "tomato_data")
    print("   ✅ Tamamlandı")
else:
    print("✅ Veri seti yerinde")

# GPU kontrolü
print(f"\n🖥️  GPU sayısı: {len(tf.config.list_physical_devices('GPU'))}")
print(f"📁 Veri seti: tomato_data/")
print(f"💾 Drive proje yolu: {drive_proje}")

In [ ]:
# ============================================================
# 2. DATAGENERATOR KURULUMU
# MobileNetV2 için: rescale=1./255 yeterli
# ============================================================

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Eğitim için: normalize + data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)

# Valid ve Test için: SADECE normalize (augmentation YOK)
valid_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Generator'ları oluştur
train_generator = train_datagen.flow_from_directory(
    "tomato_data/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

valid_generator = valid_datagen.flow_from_directory(
    "tomato_data/valid",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    "tomato_data/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("\n✅ DataGenerator'lar hazır")
print(f"   Train: {train_generator.samples} görüntü, {len(train_generator)} batch")
print(f"   Valid: {valid_generator.samples} görüntü")
print(f"   Test:  {test_generator.samples} görüntü")

In [ ]:
# ============================================================
# 3. MOBILENETV2 MODEL KURULUMU (Transfer Learning)
# ============================================================

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

print("🔨 MobileNetV2 modeli kuruluyor...")

# 1. ImageNet ağırlıklarıyla MobileNetV2'yi yükle
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,        # Son katmanı (1000 sınıf) atla
    weights='imagenet'        # Önceden eğitilmiş ağırlıklar
)

# 2. Base modeli dondur (Phase 1 için)
base_model.trainable = False

# 3. Üstüne kendi sınıflandırma katmanlarımızı ekle
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

# 4. Modeli oluştur
model = Model(inputs=base_model.input, outputs=predictions)

# 5. Compile
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Model bilgileri
print(f"\n✅ MobileNetV2 hazır")
print(f"   Toplam parametre: {model.count_params():,}")
print(f"   Toplam katman: {len(base_model.layers)}")

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"   Eğitilebilir: {trainable_params:,}")
print(f"   Donmuş: {model.count_params() - trainable_params:,}")

In [ ]:
# ============================================================
# 4. PHASE 1: FEATURE EXTRACTION (15 epoch)
# Base donmuş, sadece üst katmanlar eğitiliyor
# ============================================================

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import time

os.makedirs("models", exist_ok=True)

callbacks_phase1 = [
    ModelCheckpoint(
        'models/mobilenetv2_phase1.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    )
]

print("=" * 70)
print("🚀 PHASE 1: Feature Extraction")
print("=" * 70)
print("Süreç: Sadece üst katmanlar eğitiliyor")
print("Beklenen süre: ~50-60 dakika\n")

start = time.time()

history_phase1 = model.fit(
    train_generator,
    epochs=15,
    validation_data=valid_generator,
    callbacks=callbacks_phase1,
    verbose=1
)

elapsed = time.time() - start
print(f"\n✅ Phase 1 tamamlandı! Süre: {elapsed/60:.1f} dakika")
print(f"   En iyi val_accuracy: {max(history_phase1.history['val_accuracy']):.4f}")

In [ ]:
# ============================================================
# 5. PHASE 2 HAZIRLIK: FINE-TUNING
# Son 30 katmanı eğitilebilir yap, learning rate'i düşür
# ============================================================

# Base modelin son katmanlarını eğitilebilir yap
base_model.trainable = True

total_layers = len(base_model.layers)
print(f"MobileNetV2 toplam katman: {total_layers}")

# Son 30 katmanı aç, geri kalanı dondur
fine_tune_at = total_layers - 30

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# DİKKAT: Çok düşük learning rate (1e-5)
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\n✅ Fine-tuning için hazır")
print(f"   Eğitilebilir parametre: {trainable_params:,}")
print(f"   Donmuş katman: {fine_tune_at}")
print(f"   Açık katman: 30")

In [ ]:
# ============================================================
# 6. PHASE 2: FINE-TUNING (10 epoch)
# Son 30 katman + üst katmanlar düşük LR ile eğitiliyor
# ============================================================

callbacks_phase2 = [
    ModelCheckpoint(
        'models/mobilenetv2_final.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-8, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
]

print("=" * 70)
print("🚀 PHASE 2: Fine-Tuning")
print("=" * 70)
print("Süreç: Son 30 katman + üst katmanlar açık, düşük LR")
print("Beklenen süre: ~35-40 dakika\n")

start = time.time()

history_phase2 = model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator,
    callbacks=callbacks_phase2,
    verbose=1
)

elapsed = time.time() - start
print(f"\n✅ Phase 2 tamamlandı! Süre: {elapsed/60:.1f} dakika")
print(f"   En iyi val_accuracy: {max(history_phase2.history['val_accuracy']):.4f}")

print(f"\n📊 İyileşme:")
phase1_best = max(history_phase1.history['val_accuracy'])
phase2_best = max(history_phase2.history['val_accuracy'])
print(f"   Phase 1: {phase1_best*100:.2f}%")
print(f"   Phase 2: {phase2_best*100:.2f}%")

In [ ]:
# ============================================================
# 7. TEST SETİ DEĞERLENDİRMESİ
# ============================================================

from tensorflow.keras.models import load_model
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
import numpy as np

# En iyi modeli yükle
print("En iyi model yükleniyor...")
best_model = load_model('models/mobilenetv2_final.keras')

# Test setinde değerlendir
print("\nTest setinde değerlendiriliyor...")
test_generator.reset()
test_loss, test_accuracy = best_model.evaluate(test_generator, verbose=1)

# Tahminler
test_generator.reset()
predictions = best_model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Metrikler
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
model_size = os.path.getsize('models/mobilenetv2_final.keras') / (1024 * 1024)

# Özet rapor
print("\n" + "=" * 70)
print("📋 MOBILENETV2 ÖZET RAPORU")
print("=" * 70)
print(f"  Test Accuracy:     {test_accuracy*100:.2f}%")
print(f"  Test Loss:         {test_loss:.4f}")
print(f"  Precision:         {precision:.4f}")
print(f"  Recall:            {recall:.4f}")
print(f"  F1-Score:          {f1:.4f}")
print(f"  Model Boyutu:      {model_size:.2f} MB")
print("=" * 70)

# Sınıf bazlı detaylı rapor
class_names = list(test_generator.class_indices.keys())
print("\n📊 SINIF BAZLI PERFORMANS:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
# ============================================================
# 8. EĞİTİLMİŞ MODELİ DRIVE'A YEDEKLE
# ============================================================

# Phase 1 ve Final modelleri Drive'a kopyala
os.makedirs(f"{drive_proje}/models", exist_ok=True)

for model_file in ['mobilenetv2_phase1.keras', 'mobilenetv2_final.keras']:
    src = f"models/{model_file}"
    dst = f"{drive_proje}/models/{model_file}"
    if os.path.exists(src):
        shutil.copy(src, dst)
        size_mb = os.path.getsize(dst) / (1024*1024)
        print(f"✅ {model_file} Drive'a yedeklendi ({size_mb:.1f} MB)")

print("\n📌 Model artık Drive'da güvende.")
print(f"   Yol: {drive_proje}/models/")

## ✅ MobileNetV2 Eğitimi Tamamlandı

### Sonuçlar
- **Test Accuracy**: %92.40
- **F1-Score**: 0.9242
- **Model Boyutu**: 22.74 MB
- **Eğitim Süresi**: ~94 dakika (Phase 1 + Phase 2)

### Çıktılar
- `models/mobilenetv2_final.keras` — Eğitilmiş model
- Drive yedeği: `Domates_Projesi/models/mobilenetv2_final.keras`

### Yorum
MobileNetV2, hafif yapısı sayesinde mobil cihazlarda çalışabilir.
Performansı kabul edilebilir seviyede (%92.40) ancak bazı zor sınıflarda
(Hedef Leke gibi) ResNet50'den geride kalıyor.

### Sıradaki Adım
👉 `04_resnet50_egitim.ipynb` notebook'unu açın ve daha güçlü bir modeli deneyin.